# Predicting Colorectal Cancer CMS Subtypes with Deep Learning (MLP)

**Authors:** Serna Sürer, Luca Baldi

## 1. Introduction

Colorectal cancer is a molecularly heterogeneous disease, meaning that patients with the same cancer type can still differ strongly at the molecular level. One established way to describe this heterogeneity is the Consensus Molecular Subtype (CMS) classification, which groups colorectal cancer samples into biologically meaningful subtypes based on gene expression patterns.

In this project, we investigate whether gene expression data can be used to distinguish CMS2 samples from non-CMS2 colorectal cancer samples using a simple deep learning model. To keep the task feasible and interpretable while still using the full dataset, the analysis is formulated as a binary classification problem: CMS2 versus non-CMS2.

The model used in this notebook is a Feedforward Neural Network, also referred to as a Multi-Layer Perceptron (MLP), implemented in PyTorch. The aim is not to develop a clinically validated classifier, but to demonstrate a complete deep learning workflow for biomedical tabular data.

To test this, the gene expression data are combined with CMS subtype labels and converted into a binary target: CMS2 samples are encoded as the positive class, while CMS1, CMS3, and CMS4 samples are grouped as non-CMS2. The data are split into training, validation, and test sets, followed by scaling and feature selection to reduce the high dimensionality of the gene expression data. An MLP model is then trained on the selected features and evaluated using classification metrics such as accuracy, precision, recall, F1 score, ROC-AUC, as well as a confusion matrix and ROC curve.

## 2. Setup

This section imports the libraries required for data handling, visualization, preprocessing, model training, and evaluation. The notebook uses `pandas` and `numpy` for data processing, `scikit-learn` for preprocessing and evaluation metrics, and `PyTorch` to build and train the MLP model.

A fixed random seed is used to make the results more reproducible. The code also selects a GPU if available; otherwise, the model is trained on the CPU. Output folders for figures and results are created to keep generated files organized.

In [ ]:
# Basic libraries
from pathlib import Path
import random
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
RANDOM_STATE = 42

def set_seed(seed: int = 42) -> None:
    """Set random seeds to make results more reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

FIGURE_DIR = Path("figures")
RESULTS_DIR = Path("results")

FIGURE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

## 3. Data Loading and Preparation

In this section, the gene expression data and CMS subtype annotations are loaded and prepared for modelling. The gene expression dataset contains TCGA samples as rows and genes as columns. The subtype annotation file contains the corresponding CMS labels and selected clinical information.

After loading the files, basic checks are performed to ensure that the sample identifiers are unique and that both datasets refer to the same patients. The datasets are then merged using the TCGA `submitter_id`, and only samples that are present in both files are retained.

In [ ]:
# Data paths
EXPRESSION_FILE = r"C:\Users\serna\OneDrive\Desktop\dataset_deep_learning\TCGA_CRC_gene_expression_cpm.csv"
SUBTYPE_FILE = r"C:\Users\serna\OneDrive\Desktop\dataset_deep_learning\TCGA_CRC_subtypes.csv"

In [ ]:
# Load input files
expression_df = pd.read_csv(EXPRESSION_FILE)
subtypes_df = pd.read_csv(SUBTYPE_FILE)

print("Gene expression shape:", expression_df.shape)
print("Subtype annotation shape:", subtypes_df.shape)

In [ ]:
# Check identifiers and CMS class distribution
print("Expression submitter_id unique:", expression_df["submitter_id"].is_unique)
print("Subtype submitter_id unique:", subtypes_df["submitter_id"].is_unique)

common_samples = set(expression_df["submitter_id"]).intersection(set(subtypes_df["submitter_id"]))
print("Number of common samples:", len(common_samples))

print("\nCMS class distribution:")
display(subtypes_df["CMS"].value_counts(dropna=False))

In [ ]:
# Merge gene expression data with CMS labels
data = expression_df.merge(
    subtypes_df[["submitter_id", "CMS", "project_id", "pathologic_stage"]],
    on="submitter_id",
    how="inner"
)

print("Merged data shape:", data.shape)

## 4. Binary Target and Feature Definition

This section defines the binary classification task used in this notebook. All CMS samples are retained, but the target is converted into a binary label: CMS2 samples are encoded as the positive class (`1`), while CMS1, CMS3, and CMS4 samples are grouped as non-CMS2 and encoded as the negative class (`0`).

The class distribution is inspected to check whether the binary task is balanced or imbalanced. This is important because class imbalance can make accuracy misleading. For the baseline model, only gene expression columns are used as input features, while CMS labels and clinical metadata are excluded from the feature matrix.

In [ ]:
POSITIVE_CLASS = "CMS2"
NEGATIVE_CLASS = "non-CMS2"

binary_data = data[data["CMS"].notna()].copy()

binary_data["target"] = (binary_data["CMS"] == POSITIVE_CLASS).astype(int)
binary_data["binary_label"] = binary_data["target"].map({
    1: POSITIVE_CLASS,
    0: NEGATIVE_CLASS
})

print("Binary dataset shape:", binary_data.shape)
print("\nBinary class distribution:")
display(binary_data["binary_label"].value_counts())

In [ ]:
class_counts = binary_data["binary_label"].value_counts().reindex([POSITIVE_CLASS, NEGATIVE_CLASS])

plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)
plt.title(f"Class Distribution: {POSITIVE_CLASS} vs {NEGATIVE_CLASS}")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_distribution_binary.png", dpi=300)
plt.show()

In [ ]:
# Use only gene expression columns as model input
gene_columns = [col for col in expression_df.columns if col != "submitter_id"]


X = binary_data[gene_columns].copy()
y = binary_data["target"].astype(int).copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Number of gene features:", len(gene_columns))

## 5. Preprocessing and Feature Selection

In this section, the data are split into training, validation, and test sets. The training set is used to fit the model, the validation set is used for model selection and early stopping, and the test set is kept untouched until the final evaluation.

Because gene expression data contains many more features than samples, preprocessing is applied before model training. First, constant genes are removed using a variance filter. Then, the most informative genes are selected using ANOVA F-scores. Finally, the selected features are scaled. All preprocessing steps are fitted only on the training data to avoid data leakage.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Training set:", X_train.shape, y_train.value_counts().to_dict())
print("Validation set:", X_val.shape, y_val.value_counts().to_dict())
print("Test set:", X_test.shape, y_test.value_counts().to_dict())

In [ ]:
# Remove constant genes before feature selection
variance_filter = VarianceThreshold(threshold=0.0)

X_train_var = variance_filter.fit_transform(X_train)
X_val_var = variance_filter.transform(X_val)
X_test_var = variance_filter.transform(X_test)

remaining_gene_names = X_train.columns[variance_filter.get_support()]

# Select top genes based on ANOVA F-score
N_SELECTED_GENES = 500
k = min(N_SELECTED_GENES, X_train_var.shape[1])

feature_selector = SelectKBest(score_func=f_classif, k=k)

X_train_selected = feature_selector.fit_transform(X_train_var, y_train)
X_val_selected = feature_selector.transform(X_val_var)
X_test_selected = feature_selector.transform(X_test_var)

selected_gene_names = remaining_gene_names[feature_selector.get_support()].tolist()

# Scale selected features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_val_scaled = scaler.transform(X_val_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("Selected number of genes:", len(selected_gene_names))
print("Processed training shape:", X_train_scaled.shape)

selected_genes_df = pd.DataFrame({"selected_gene": selected_gene_names})
selected_genes_df.to_csv(RESULTS_DIR / "selected_genes.csv", index=False)

## 6. Exploratory Visualization

PCA is used as an exploratory visualization to inspect whether CMS2 and non-CMS2 samples show visible separation based on the selected gene expression features. The high-dimensional feature matrix is reduced to two principal components so that the samples can be plotted in a two-dimensional space.

This visualization is not used as input for the MLP model. It only helps to understand whether the selected gene expression features contain a visible signal that may support binary classification.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

X_pca = np.vstack([X_train_pca, X_val_pca, X_test_pca])

plot_labels = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)
plot_classes = plot_labels.map({
    1: POSITIVE_CLASS,
    0: NEGATIVE_CLASS
})

plt.figure(figsize=(6, 5))
for class_name in [POSITIVE_CLASS, NEGATIVE_CLASS]:
    mask = plot_classes == class_name
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=class_name,
        alpha=0.8
    )

plt.title("PCA of Selected Gene Expression Features")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_binary_selected_genes.png", dpi=300)
plt.show()

The PCA plot shows partial separation between CMS2 and non-CMS2 samples, mainly along the first principal component. Compared with the previous CMS1 vs CMS2 setting, the overlap is stronger because the non-CMS2 group combines several biologically different CMS subtypes. Therefore, PCA is used only as an exploratory visualization, while the final conclusion should be based on the independent test evaluation.

## 7. MLP Modelling

In this section, the processed gene expression data are prepared for PyTorch and used to train a binary MLP classifier. The scaled feature matrices are converted into tensors and loaded in mini-batches using DataLoaders.

The model is a simple Feedforward Neural Network / Multi-Layer Perceptron with two hidden layers, ReLU activation functions, dropout regularization, and one output neuron for binary classification. The model is trained using binary cross-entropy loss with logits and the Adam optimizer. To reduce overfitting, L2 regularization and early stopping based on the validation loss are applied.

In [ ]:
BATCH_SIZE = 16
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15

def make_dataloader(X_array, y_series, batch_size=16, shuffle=False):
    """Create a PyTorch DataLoader from feature and label arrays."""
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    y_tensor = torch.tensor(y_series.values, dtype=torch.float32).view(-1, 1)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_dataloader(X_train_scaled, y_train, BATCH_SIZE, shuffle=True)
val_loader = make_dataloader(X_val_scaled, y_val, BATCH_SIZE, shuffle=False)
test_loader = make_dataloader(X_test_scaled, y_test, BATCH_SIZE, shuffle=False)

In [ ]:
class MLPClassifier(nn.Module):
    """Feedforward neural network for binary CMS classification."""

    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.3):
        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train the model for one epoch and return the average loss."""
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(dataloader.dataset)


def evaluate_loss(model, dataloader, criterion, device):
    """Evaluate the model loss without updating weights."""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(dataloader.dataset)


def predict_probabilities(model, dataloader, device):
    """Return predicted probabilities for the positive class."""
    model.eval()
    probabilities = []

    with torch.no_grad():
        for X_batch, _ in dataloader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = torch.sigmoid(logits)
            probabilities.extend(probs.cpu().numpy().ravel())

    return np.array(probabilities)

In [ ]:
input_dim = X_train_scaled.shape[1]

model = MLPClassifier(
    input_dim=input_dim,
    hidden_dims=(128, 64),
    dropout=0.3,
).to(device)

print(model)

# Handle possible class imbalance with pos_weight
n_positive = int(y_train.sum())
n_negative = int(len(y_train) - y_train.sum())
pos_weight_value = n_negative / max(n_positive, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = {
    "train_loss": [],
    "val_loss": [],
}

best_val_loss = np.inf
best_model_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate_loss(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best validation loss: {best_val_loss:.4f}")
        break

model.load_state_dict(best_model_state)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "training_validation_loss.png", dpi=300)
plt.show()

The training loss decreases quickly, while the validation loss increases after the first epochs. This indicates overfitting: the model learns the training data well but does not generalize equally well to the validation set. Therefore, early stopping is used to keep the model state with the lowest validation loss.

## 8. Final Model Evaluation

The final model is evaluated on the independent test set, which was not used during training or model selection. This provides a more realistic estimate of how well the model generalizes to unseen samples.

Performance is assessed using accuracy, balanced accuracy, precision, recall, F1-score, ROC-AUC, a confusion matrix, and a ROC curve. These metrics provide a more complete evaluation than accuracy alone, especially if the class distribution is imbalanced.

In [ ]:
test_probabilities = predict_probabilities(model, test_loader, device)
test_predictions = (test_probabilities >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(RESULTS_DIR / "baseline_mlp_test_metrics.csv", index=False)

display(metrics_df)

print("Classification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=[NEGATIVE_CLASS, POSITIVE_CLASS],
    zero_division=0,
))

In [ ]:
cm = confusion_matrix(y_test, test_predictions)

display_labels = [NEGATIVE_CLASS, POSITIVE_CLASS]

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=display_labels,
)

disp.plot(values_format="d")
plt.title("Confusion Matrix: CMS2 vs non-CMS2")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "confusion_matrix_binary_mlp.png", dpi=300)
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    test_probabilities,
    name="MLP",
)

plt.title("ROC Curve: CMS2 vs non-CMS2")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "roc_curve_binary_mlp.png", dpi=300)
plt.show()

The final test results show strong performance for distinguishing CMS2 from non-CMS2 samples. The confusion matrix shows that most samples were classified correctly, with only a small number of false predictions. The ROC-AUC score also indicates good separation between the two classes. However, because the dataset is relatively small and the validation curve showed signs of overfitting, the results should be interpreted carefully.

## 9. Robustness Experiment

The final model evaluation is based on one train/validation/test split. To check whether the result is stable, an additional robustness experiment is performed using different random splits and different numbers of selected genes.

For each setting, the same preprocessing and MLP training workflow is repeated. The aim is to evaluate whether performance remains consistent or whether it strongly depends on the selected split or the number of input genes.

In [ ]:
def train_and_evaluate_for_gene_count(
    n_genes,
    seed,
    hidden_dims=(128, 64),
    dropout=0.3,
    max_epochs=60,
    patience=10,
):
    """Train and evaluate the MLP for one feature count and one random split."""
    set_seed(seed)

    # Train / validation / test split
    X_train_split, X_temp_split, y_train_split, y_temp_split = train_test_split(
        X,
        y,
        test_size=0.30,
        stratify=y,
        random_state=seed,
    )

    X_val_split, X_test_split, y_val_split, y_test_split = train_test_split(
        X_temp_split,
        y_temp_split,
        test_size=0.50,
        stratify=y_temp_split,
        random_state=seed,
    )
    
    variance_filter = VarianceThreshold(threshold=0.0)
    X_train_var = variance_filter.fit_transform(X_train_split)
    X_val_var = variance_filter.transform(X_val_split)
    X_test_var = variance_filter.transform(X_test_split)
    k = min(n_genes, X_train_var.shape[1])
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_selected = selector.fit_transform(X_train_var, y_train_split)
    X_val_selected = selector.transform(X_val_var)
    X_test_selected = selector.transform(X_test_var)

    # Feature selection: fit only on training data
    k = min(n_genes, X_train_split.shape[1])
    selector = SelectKBest(score_func=f_classif, k=k)

    X_train_selected = selector.fit_transform(X_train_split, y_train_split)
    X_val_selected = selector.transform(X_val_split)
    X_test_selected = selector.transform(X_test_split)

    # Scaling: fit only on training data
    scaler = StandardScaler()
    X_train_scaled_split = scaler.fit_transform(X_train_selected)
    X_val_scaled_split = scaler.transform(X_val_selected)
    X_test_scaled_split = scaler.transform(X_test_selected)

    # DataLoaders
    train_loader_split = make_dataloader(
        X_train_scaled_split,
        y_train_split,
        BATCH_SIZE,
        shuffle=True,
    )

    val_loader_split = make_dataloader(
        X_val_scaled_split,
        y_val_split,
        BATCH_SIZE,
        shuffle=False,
    )

    test_loader_split = make_dataloader(
        X_test_scaled_split,
        y_test_split,
        BATCH_SIZE,
        shuffle=False,
    )

    # Model
    model_split = MLPClassifier(
        input_dim=X_train_scaled_split.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    # Class imbalance handling
    n_positive = int(y_train_split.sum())
    n_negative = int(len(y_train_split) - y_train_split.sum())
    pos_weight_value = n_negative / max(n_positive, 1)
    pos_weight_split = torch.tensor(
        [pos_weight_value],
        dtype=torch.float32,
    ).to(device)

    criterion_split = nn.BCEWithLogitsLoss(pos_weight=pos_weight_split)

    optimizer_split = torch.optim.Adam(
        model_split.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_loss = np.inf
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        train_one_epoch(
            model_split,
            train_loader_split,
            criterion_split,
            optimizer_split,
            device,
        )

        val_loss = evaluate_loss(
            model_split,
            val_loader_split,
            criterion_split,
            device,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model_split.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model_split.load_state_dict(best_state)

    # Test evaluation
    test_probabilities = predict_probabilities(
        model_split,
        test_loader_split,
        device,
    )

    test_predictions = (test_probabilities >= 0.5).astype(int)

    return {
        "seed": seed,
        "n_genes": n_genes,
        "test_accuracy": accuracy_score(y_test_split, test_predictions),
        "test_balanced_accuracy": balanced_accuracy_score(y_test_split, test_predictions),
        "test_precision": precision_score(y_test_split, test_predictions, zero_division=0),
        "test_recall": recall_score(y_test_split, test_predictions, zero_division=0),
        "test_f1": f1_score(y_test_split, test_predictions, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test_split, test_probabilities),
        "best_val_loss": best_val_loss,
        "test_size": len(y_test_split),
        "test_positive_samples": int(y_test_split.sum()),
        "test_negative_samples": int(len(y_test_split) - y_test_split.sum()),
    }

In [ ]:
# Different numbers of selected genes
gene_counts = [10, 25, 50, 100, 500]

# Several random splits
seeds = range(20)

robustness_results = []

for n_genes in gene_counts:
    for seed in seeds:
        result = train_and_evaluate_for_gene_count(
            n_genes=n_genes,
            seed=seed,
        )
        robustness_results.append(result)

robustness_results_df = pd.DataFrame(robustness_results)
robustness_results_df.to_csv(
    RESULTS_DIR / "robustness_feature_count_results.csv",
    index=False,
)

display(robustness_results_df.head())

In [ ]:
summary_results_df = (
    robustness_results_df
    .groupby("n_genes")
    .agg(
        accuracy_mean=("test_accuracy", "mean"),
        accuracy_std=("test_accuracy", "std"),
        balanced_accuracy_mean=("test_balanced_accuracy", "mean"),
        balanced_accuracy_std=("test_balanced_accuracy", "std"),
        f1_mean=("test_f1", "mean"),
        f1_std=("test_f1", "std"),
        roc_auc_mean=("test_roc_auc", "mean"),
        roc_auc_std=("test_roc_auc", "std"),
    )
    .reset_index()
)

summary_results_df.to_csv(
    RESULTS_DIR / "robustness_feature_count_summary.csv",
    index=False,
)

display(summary_results_df)

In [ ]:
plot_gene_counts = sorted(summary_results_df["n_genes"].unique())

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

plt.figure(figsize=(8, 5))

plt.errorbar(
    summary_results_df["n_genes"],
    summary_results_df["f1_mean"],
    yerr=summary_results_df["f1_std"],
    marker="o",
    linewidth=2.2,
    capsize=5,
    label="F1-score",
)

plt.errorbar(
    summary_results_df["n_genes"],
    summary_results_df["balanced_accuracy_mean"],
    yerr=summary_results_df["balanced_accuracy_std"],
    marker="s",
    linewidth=2.2,
    capsize=5,
    label="Balanced accuracy",
)

plt.xscale("log")
plt.xticks(plot_gene_counts, plot_gene_counts)
plt.ylim(0.75, 1.03)

plt.title("Model Performance by Number of Selected Genes")
plt.xlabel("Number of selected genes")
plt.ylabel("Test performance")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "performance_by_number_of_genes_presentation.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

boxplot_data = [
    robustness_results_df.loc[
        robustness_results_df["n_genes"] == n_genes,
        "test_f1",
    ]
    for n_genes in gene_counts
]

plt.boxplot(
    boxplot_data,
    labels=gene_counts,
    showmeans=True,
)

plt.title("Test F1-score Variability Across Random Splits")
plt.xlabel("Number of selected genes")
plt.ylabel("Test F1-score")
plt.ylim(0.80, 1.02)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "f1_score_variability_boxplot_presentation.png", dpi=300)
plt.show()

In [ ]:
plot_gene_counts = sorted(robustness_results_df["n_genes"].unique())

plt.figure(figsize=(8, 5))

for n_genes in plot_gene_counts:
    subset = robustness_results_df[robustness_results_df["n_genes"] == n_genes]
    
    x_values = np.exp(
        np.random.normal(
            loc=np.log(n_genes),
            scale=0.035,
            size=len(subset),
        )
    )
    
    plt.scatter(
        x_values,
        subset["test_f1"],
        alpha=0.45,
        s=35,
    )

plt.plot(
    summary_results_df["n_genes"],
    summary_results_df["f1_mean"],
    marker="o",
    linewidth=2.5,
    label="Mean F1-score",
)

plt.xscale("log")
plt.xticks(plot_gene_counts, plot_gene_counts)
plt.ylim(0.75, 1.03)

plt.title("F1-score Variability Across Random Splits")
plt.xlabel("Number of selected genes")
plt.ylabel("Test F1-score")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "f1_score_random_splits_presentation.png", dpi=300)
plt.show()

### Optional: Save Model

The trained model and preprocessing objects can be saved if the workflow should be reused later. This step is optional for the current project.

In [ ]:
import pickle

torch.save(model.state_dict(), RESULTS_DIR / "baseline_mlp_model.pt")

with open(RESULTS_DIR / "preprocessing_objects.pkl", "wb") as file:
    pickle.dump(
        {
            "selected_gene_names": selected_gene_names,
            "variance_filter": variance_filter,
            "feature_selector": feature_selector,
            "scaler": scaler,
            "positive_class": POSITIVE_CLASS,
            "negative_class": NEGATIVE_CLASS,
        },
        file,
    )

print("Model and preprocessing objects saved in:", RESULTS_DIR.resolve())

## 10. Discussion

This project investigated whether a simple PyTorch-based MLP can distinguish CMS2 colorectal cancer samples from non-CMS2 samples using gene expression data. Instead of restricting the analysis to only two CMS classes, all available CMS-labelled samples were retained and converted into a binary classification task. CMS2 samples were encoded as the positive class, while CMS1, CMS3, and CMS4 samples were grouped as non-CMS2.

The PCA visualization showed partial separation between CMS2 and non-CMS2 samples based on the selected gene expression features. However, the two groups were not completely separated, which is expected because the non-CMS2 class combines several biologically different CMS subtypes. Therefore, the PCA plot was used only as an exploratory visualization and not as evidence of final model performance.

The MLP achieved strong performance on the independent test set, with high accuracy, balanced accuracy, F1 score, and ROC-AUC. The confusion matrix also showed that most CMS2 and non-CMS2 samples were classified correctly. This suggests that the selected gene expression features contain relevant information for distinguishing CMS2 from the remaining CMS subtypes.

At the same time, the training curves showed signs of overfitting. The training loss decreased strongly, while the validation loss increased after the first epochs. This indicates that the model learned the training data very well, but generalization to unseen validation samples was more difficult. Early stopping was therefore important to keep the model state with the lowest validation loss.

Several limitations should be considered. First, the dataset is relatively small compared with the very large number of gene expression features. Second, the non-CMS2 class is heterogeneous because it combines CMS1, CMS3, and CMS4 samples into one group. Third, feature selection was necessary to reduce dimensionality, but the selected genes may depend on the specific train/test split. For this reason, the robustness experiment was included to check whether performance remains stable across different random splits and different numbers of selected genes.

Overall, the results show that the MLP can learn meaningful patterns from gene expression data for the CMS2 vs non-CMS2 classification task. However, the model should not be interpreted as a clinically validated classifier. Further validation on an external dataset would be required before drawing stronger biological or clinical conclusions.

## 11. Conclusion

This notebook demonstrates a complete deep learning workflow for binary CMS subtype prediction using colorectal cancer gene expression data. The original CMS labels were transformed into a CMS2 vs non-CMS2 classification task, allowing all available CMS-labelled samples to be used while keeping the problem binary.

The workflow included data loading, label preparation, train/validation/test splitting, removal of constant genes, feature selection, scaling, exploratory PCA visualization, MLP training, final test evaluation, and a robustness experiment. The final model achieved strong test performance, suggesting that CMS2 samples can be distinguished from non-CMS2 samples based on selected gene expression features.

However, the results should be interpreted carefully because the dataset is small, the input dimensionality is high, and the non-CMS2 class contains multiple different CMS subtypes. The validation loss also indicated overfitting, highlighting the importance of early stopping and independent test evaluation.

In conclusion, the project shows that a simple MLP can be applied successfully to this biomedical classification problem, but further experiments with external validation data, alternative model architectures, and comparison to simpler baseline models would be needed to assess the robustness and clinical relevance of the approach.

## Use of AI

AI tools were used to clarify questions and to support understanding of concepts before starting the project. Furthermore, AI was used for code debugging and for improving English grammar and clarity. All analytical decisions, interpretations, and conclusions were made independently by the author.